# 九曜 LF01 影片生成測試：Wan2.1 VACE 1.3B（參考圖→影片），A05 f2 齊衡烈開門打量机遙

In [ ]:
import subprocess
print(subprocess.run('pip uninstall -y -q torchao; pip install -q -U "diffusers>=0.35" transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece 2>&1 | tail -2; nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True, capture_output=True, text=True).stdout)

In [ ]:
import base64, io, json, time, torch, numpy as np
from PIL import Image
REFS=json.loads(r'''{"\u9f4a\u8861\u70c8": "...", "\u673a\u9059": "...", "\u7b2c\u4e03\u5ba4": "..."}''')
img={k:Image.open(io.BytesIO(base64.b64decode(v))).convert('RGB') for k,v in REFS.items()}
PROMPT="Semi-realistic stylized 3D Chinese donghua animation, cinematic. Inside an old dark-wood dormitory common room with lattice windows and warm morning light stripes on the polished floor. A wooden door is pulled open from inside: a red spiky-haired young man in dark bronze-trimmed sleeveless armour (the first reference) leans one arm on the door frame, sizes up the newcomer from head to toe, then grins widely and slaps his own right upper arm. The newcomer, a black-haired young man in a plain dark-grey robe with a rolled cloth bundle on his back (the second reference), stands at the threshold, a little stiff and polite. Medium two-shot, side profiles facing each other, natural body motion, steady camera with a slow push in."
NEG="2D, anime cel shading, cartoon, low quality, blurry, distorted face, extra limbs, extra fingers, deformed hands, text, subtitles, watermark, static image, overexposed, character changes appearance, duplicated person"
print({k:v.size for k,v in img.items()})

In [ ]:
from diffusers import AutoencoderKLWan, WanVACEPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
mid='Wan-AI/Wan2.1-VACE-1.3B-diffusers'
vae=AutoencoderKLWan.from_pretrained(mid, subfolder='vae', torch_dtype=torch.float32)
pipe=WanVACEPipeline.from_pretrained(mid, vae=vae, torch_dtype=torch.float16)
pipe.scheduler=UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.enable_model_cpu_offload()
print('loaded')

In [ ]:
def run(name, refs, steps=30):
    t=time.time()
    frames=pipe(prompt=PROMPT, negative_prompt=NEG, reference_images=refs, height=480, width=832, num_frames=81,
                num_inference_steps=steps, guidance_scale=5.0, generator=torch.Generator('cpu').manual_seed(7)).frames[0]
    arr=np.asarray(frames); print(name, 'shape', arr.shape, 'mean', float(np.nanmean(arr)), 'nan', bool(np.isnan(arr).any()), f'{time.time()-t:.0f}s', flush=True)
    path=f'/kaggle/working/{name}.mp4'; export_to_video(frames, path, fps=16)
    print('@@MP4', name+'.mp4', base64.b64encode(open(path,'rb').read()).decode(), flush=True)
run('wan_a_chars', [img['齊衡烈'], img['机遙']])
run('wan_b_chars_room', [img['齊衡烈'], img['机遙'], img['第七室']])